# Brain Tumor Detection with YOLOv8

This notebook prepares a YOLO-format detection dataset, checks image/label pairs and annotations, visualizes boxes, trains YOLOv8, evaluates on a held-out test set, and saves artifacts.

> **Before running:** edit the paths and class names in Section 1. A YOLO label row must be `class_id x_center y_center width height`, with normalized coordinates in the range 0–1. Classification-only folders do not contain localization labels and cannot be used directly for detection training.

**Suggested environment:** Python 3.10–3.12 with `ultralytics`, `opencv-python`, `matplotlib`, `pandas`, `PyYAML`, `scikit-learn`, and `tqdm`.

## 1. Install / import dependencies

In [ ]:
# If needed, uncomment in a notebook environment:
# %pip install -U ultralytics opencv-python matplotlib pandas pyyaml scikit-learn tqdm

from pathlib import Path
import os
import random
import shutil
import warnings
from collections import Counter

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from ultralytics import YOLO
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

## 2. Configure source paths, classes, and training settings

The source may be a single image folder plus a single annotation folder. If your dataset already has train/valid/test folders, it is usually better to preserve those splits instead of re-splitting them.

In [ ]:
# EDIT THESE PATHS:
# Example flat structure:
# raw_dataset/
#   images/  (image files)
#   labels/  (matching .txt files)
SOURCE_IMAGES = Path("/path/to/raw_dataset/images")
SOURCE_LABELS = Path("/path/to/raw_dataset/labels")

# Replace with the actual class names in your annotation dataset, ordered by class ID.
# Example only: if every box is simply a tumor, use ["tumor"].
CLASS_NAMES = ["tumor"]

# Prepared dataset destination
DATASET_DIR = Path("./brain_tumor_yolo_dataset")

# Training configuration
MODEL_WEIGHTS = "yolov8n.pt"  # light model; consider yolov8s.pt if resources allow
IMAGE_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 8                 # reduce to 4 or 2 if memory is limited
WORKERS = 2
DEVICE = "0" if __import__("torch").cuda.is_available() else "cpu"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
assert len(CLASS_NAMES) > 0, "CLASS_NAMES must contain at least one class."
assert SOURCE_IMAGES.exists(), f"Image directory not found: {SOURCE_IMAGES}"
assert SOURCE_LABELS.exists(), f"Label directory not found: {SOURCE_LABELS}"

print("Device:", DEVICE)
print("Classes:", dict(enumerate(CLASS_NAMES)))

## 3. Inventory images and annotation files

This checks image readability, detects missing label/image pairs by filename stem, and records the files found. The stem-based matching means `case_01.jpg` matches `case_01.txt`.

In [ ]:
image_paths = sorted(
    p for p in SOURCE_IMAGES.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)
label_paths = sorted(p for p in SOURCE_LABELS.rglob("*.txt") if p.is_file())

# Use file stems as matching keys. If two files share a stem in different subfolders,
# this notebook flags that ambiguity rather than silently choosing one.
images_by_stem = {}
labels_by_stem = {}
for p in image_paths:
    images_by_stem.setdefault(p.stem, []).append(p)
for p in label_paths:
    labels_by_stem.setdefault(p.stem, []).append(p)

duplicate_image_stems = {k:v for k,v in images_by_stem.items() if len(v) > 1}
duplicate_label_stems = {k:v for k,v in labels_by_stem.items() if len(v) > 1}
image_stems, label_stems = set(images_by_stem), set(labels_by_stem)

print(f"Images discovered: {len(image_paths)}")
print(f"TXT annotations discovered: {len(label_paths)}")
print(f"Images without a matching TXT: {len(image_stems - label_stems)}")
print(f"TXT files without a matching image: {len(label_stems - image_stems)}")
print(f"Duplicate image stems: {len(duplicate_image_stems)}")
print(f"Duplicate label stems: {len(duplicate_label_stems)}")

if duplicate_image_stems:
    print("Duplicate image stems (inspect these):", list(duplicate_image_stems)[:10])
if duplicate_label_stems:
    print("Duplicate label stems (inspect these):", list(duplicate_label_stems)[:10])

unmatched_images = sorted(image_stems - label_stems)
unmatched_labels = sorted(label_stems - image_stems)

## 4. Validate image integrity and YOLO annotation format

For each label row, the class ID must be an integer from `0` to `len(CLASS_NAMES)-1`; the four box values must be numeric, finite, and normalized between 0 and 1. Width and height must be greater than zero. Empty label files are valid for images with no objects in general object detection datasets, but this brain-tumor workflow excludes them by default; change `ALLOW_EMPTY_LABELS` if your task intentionally includes negative images.

In [ ]:
ALLOW_EMPTY_LABELS = False

def read_image_info(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        return None
    h, w = image.shape[:2]
    return {"image": image, "width": w, "height": h, "channels": image.shape[2]}

def validate_yolo_label(label_path, num_classes, allow_empty=False):
    errors = []
    try:
        text = label_path.read_text(encoding="utf-8").strip()
    except Exception as exc:
        return False, [f"Cannot read label: {exc}"], []

    if not text:
        if allow_empty:
            return True, [], []
        return False, ["Empty label file (not allowed by current setting)"], []

    parsed = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        parts = line.split()
        if len(parts) != 5:
            errors.append(f"Line {line_no}: expected 5 fields, got {len(parts)}")
            continue
        try:
            cls_float = float(parts[0])
            coords = [float(x) for x in parts[1:]]
        except ValueError:
            errors.append(f"Line {line_no}: non-numeric value")
            continue

        if not cls_float.is_integer():
            errors.append(f"Line {line_no}: class ID must be an integer")
            continue
        cls_id = int(cls_float)
        if not (0 <= cls_id < num_classes):
            errors.append(f"Line {line_no}: class ID {cls_id} outside 0..{num_classes-1}")
        if not np.all(np.isfinite(coords)):
            errors.append(f"Line {line_no}: coordinates must be finite")
        x, y, w, h = coords
        if not all(0 <= v <= 1 for v in coords):
            errors.append(f"Line {line_no}: coordinates must be normalized to [0, 1]")
        if w <= 0 or h <= 0:
            errors.append(f"Line {line_no}: box width and height must be > 0")
        if 0 <= cls_id < num_classes and np.all(np.isfinite(coords)) and 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1:
            parsed.append((cls_id, x, y, w, h))
    return len(errors) == 0, errors, parsed

In [ ]:
audit_rows = []
valid_candidates = []

# Only use unique stems to avoid ambiguous image-label pairing.
for stem in tqdm(sorted(image_stems & label_stems), desc="Auditing pairs"):
    if len(images_by_stem[stem]) != 1 or len(labels_by_stem[stem]) != 1:
        audit_rows.append({"stem": stem, "status": "ambiguous duplicate stem", "errors": "duplicate filename stem"})
        continue

    image_path = images_by_stem[stem][0]
    label_path = labels_by_stem[stem][0]
    info = read_image_info(image_path)
    if info is None:
        audit_rows.append({"stem": stem, "image": str(image_path), "label": str(label_path),
                           "status": "invalid image", "errors": "OpenCV could not decode image"})
        continue

    ok, errors, parsed = validate_yolo_label(label_path, len(CLASS_NAMES), ALLOW_EMPTY_LABELS)
    if not ok:
        audit_rows.append({"stem": stem, "image": str(image_path), "label": str(label_path),
                           "status": "invalid annotation", "errors": "; ".join(errors)})
        continue

    valid_candidates.append({
        "stem": stem, "image_path": image_path, "label_path": label_path,
        "width": info["width"], "height": info["height"], "channels": info["channels"],
        "num_boxes": len(parsed), "classes": [b[0] for b in parsed]
    })
    audit_rows.append({"stem": stem, "image": str(image_path), "label": str(label_path),
                       "status": "valid", "errors": ""})

audit_df = pd.DataFrame(audit_rows)
display(audit_df["status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))
display(audit_df[audit_df["status"] != "valid"].head(30))
print("Valid image-label pairs:", len(valid_candidates))

## 5. Preview valid images with bounding boxes

YOLO coordinates are normalized center coordinates and box width/height. The plotting helper converts them to pixel coordinates. Inspect this sample carefully before training: boxes should tightly surround the intended tumor, and class names must agree with the dataset documentation.

In [ ]:
def load_boxes(label_path):
    ok, errors, parsed = validate_yolo_label(label_path, len(CLASS_NAMES), allow_empty=True)
    if not ok:
        return []
    return parsed

def show_annotated_pair(image_path, label_path, ax=None):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Unreadable image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    H, W = image.shape[:2]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image)
    for cls_id, xc, yc, bw, bh in load_boxes(label_path):
        x1 = (xc - bw / 2) * W
        y1 = (yc - bh / 2) * H
        rect = plt.Rectangle((x1, y1), bw * W, bh * H, fill=False, linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 4), CLASS_NAMES[cls_id],
                color="white", fontsize=9, bbox=dict(facecolor="red", alpha=0.7, pad=2))
    ax.set_title(Path(image_path).name)
    ax.axis("off")
    return ax

if not valid_candidates:
    print("No valid pairs found. Recheck paths, file names, class names, and label format.")
else:
    sample = random.sample(valid_candidates, min(6, len(valid_candidates)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax in axes.flat:
        ax.axis("off")
    for ax, item in zip(axes.flat, sample):
        show_annotated_pair(item["image_path"], item["label_path"], ax=ax)
    plt.tight_layout()
    plt.show()

## 6. Explore image dimensions, box counts, and class balance

In [ ]:
valid_df = pd.DataFrame([{
    "stem": x["stem"], "width": x["width"], "height": x["height"],
    "channels": x["channels"], "num_boxes": x["num_boxes"]
} for x in valid_candidates])
display(valid_df.describe(include="all"))

if not valid_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    valid_df["width"].hist(ax=axes[0], bins=20)
    axes[0].set_title("Image width distribution")
    axes[0].set_xlabel("Width (pixels)")
    valid_df["height"].hist(ax=axes[1], bins=20)
    axes[1].set_title("Image height distribution")
    axes[1].set_xlabel("Height (pixels)")
    plt.tight_layout()
    plt.show()

class_counts = Counter()
for item in valid_candidates:
    for cls_id in item["classes"]:
        class_counts[cls_id] += 1
class_balance = pd.DataFrame([
    {"class_id": i, "class_name": name, "box_count": class_counts.get(i, 0)}
    for i, name in enumerate(CLASS_NAMES)
])
display(class_balance)
class_balance.plot(x="class_name", y="box_count", kind="bar", legend=False, rot=30,
                   title="Bounding boxes per class")
plt.ylabel("Number of boxes")
plt.tight_layout()
plt.show()

## 7. Split valid pairs and create YOLO folder structure

This creates a deterministic 70% / 15% / 15% split at the **image level**, then copies each image and its matching `.txt` file into corresponding image/label folders. Do not run this split if you already have a prescribed split or related slices from the same patient must be grouped together; in medical imaging, patient-level splitting is preferable to avoid leakage.

In [ ]:
# Set True only if you want to remove and rebuild DATASET_DIR.
REBUILD_DATASET = True

if REBUILD_DATASET and DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ("train", "valid", "test"):
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

items = valid_candidates.copy()
random.Random(SEED).shuffle(items)
n = len(items)
n_train = int(n * 0.70)
n_valid = int(n * 0.15)
splits = {
    "train": items[:n_train],
    "valid": items[n_train:n_train+n_valid],
    "test": items[n_train+n_valid:]
}

copy_records = []
for split_name, split_items in splits.items():
    for item in tqdm(split_items, desc=f"Copying {split_name}"):
        # Keep the original image extension; labels use the same stem with .txt.
        dst_image = DATASET_DIR / "images" / split_name / item["image_path"].name
        dst_label = DATASET_DIR / "labels" / split_name / f"{item['stem']}.txt"
        shutil.copy2(item["image_path"], dst_image)
        shutil.copy2(item["label_path"], dst_label)
        copy_records.append({"split": split_name, "stem": item["stem"],
                             "image": str(dst_image), "label": str(dst_label)})

split_df = pd.DataFrame(copy_records)
display(split_df.groupby("split").size().rename("image_label_pairs").to_frame())
print("Dataset directory:", DATASET_DIR.resolve())

## 8. Verify split integrity and counts

In [ ]:
for split_name in ("train", "valid", "test"):
    imgs = [p for p in (DATASET_DIR / "images" / split_name).iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    labels = list((DATASET_DIR / "labels" / split_name).glob("*.txt"))
    image_stems_split = {p.stem for p in imgs}
    label_stems_split = {p.stem for p in labels}
    print(f"{split_name:>5}: images={len(imgs)}, labels={len(labels)}, "
          f"missing labels={len(image_stems_split-label_stems_split)}, "
          f"orphan labels={len(label_stems_split-image_stems_split)}")
    assert image_stems_split == label_stems_split, f"Pair mismatch in {split_name}"

## 9. Write `data.yaml`

Ultralytics expects paths to image directories and class names indexed by class ID. The label directories are inferred from the standard `images/` and `labels/` structure.

In [ ]:
data_yaml_path = DATASET_DIR / "data.yaml"
data_config = {
    "path": str(DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(CLASS_NAMES),
    "names": {i: name for i, name in enumerate(CLASS_NAMES)}
}
with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, sort_keys=False, allow_unicode=True)

print(data_yaml_path.read_text())

## 10. Train YOLOv8

Start with `yolov8n.pt` for limited hardware. A pretrained checkpoint is recommended. Augmentation values below are examples, not universal best values; keep geometric augmentation medically plausible and inspect transformed samples if you customize it. Ultralytics applies these training augmentations to training data, not to validation/test data.

In [ ]:
model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    patience=10,
    pretrained=True,
    optimizer="auto",
    lr0=0.01,
    cos_lr=True,
    close_mosaic=10,
    hsv_h=0.0,       # MRI is grayscale; avoid artificial hue changes
    hsv_s=0.0,
    hsv_v=0.2,       # modest brightness/value variation
    degrees=5.0,     # modest rotation
    translate=0.05,
    scale=0.2,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,      # only if left-right flipping is valid for your task
    flipud=0.0,      # avoid vertical flips unless clinically justified
    mosaic=0.5,
    mixup=0.0,
    project="runs/brain_tumor",
    name="yolov8n_experiment",
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
print("Run directory:", RUN_DIR.resolve())
print("Best weights exist:", BEST_WEIGHTS.exists(), BEST_WEIGHTS)

## 11. Review training curves and diagnostics

In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    history = pd.read_csv(results_csv)
    history.columns = history.columns.str.strip()
    display(history.tail())
    metric_cols = [c for c in history.columns if any(k in c.lower() for k in
                   ["precision", "recall", "map50", "map75", "map50-95", "box_loss", "cls_loss", "dfl_loss"])]
    if metric_cols:
        history.plot(x=history.columns[0], y=metric_cols, subplots=True,
                     layout=(-1, 2), figsize=(12, 3 * ((len(metric_cols)+1)//2)),
                     legend=False, title="Training history")
        plt.tight_layout()
        plt.show()
else:
    print("results.csv not found yet. Confirm training completed and RUN_DIR is correct.")

## 12. Evaluate on the held-out test set

Use the best checkpoint and explicitly pass the test split. For object detection, plain classification **accuracy** is not the main metric. Report precision, recall, mAP@0.5, and mAP@0.5:0.95; these measure detection quality and localization under IoU thresholds.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
test_metrics = best_model.val(
    data=str(data_yaml_path),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project="runs/brain_tumor",
    name="test_evaluation",
    exist_ok=True
)

box_metrics = test_metrics.box
summary_metrics = {
    "precision_mean": float(box_metrics.mp),
    "recall_mean": float(box_metrics.mr),
    "mAP50": float(box_metrics.map50),
    "mAP50_95": float(box_metrics.map),
}
display(pd.DataFrame([summary_metrics]).T.rename(columns={0: "value"}))

## 13. Per-class metrics and interpretation

In [ ]:
per_class_map = getattr(box_metrics, "maps", None)
if per_class_map is not None and len(per_class_map) == len(CLASS_NAMES):
    per_class_df = pd.DataFrame({
        "class_id": list(range(len(CLASS_NAMES))),
        "class_name": CLASS_NAMES,
        "mAP50_95": [float(x) for x in per_class_map]
    })
    display(per_class_df)
else:
    print("Per-class mAP is not available in the expected format for this Ultralytics version.")

print("Interpretation guide:")
print("- Precision: of predicted boxes, how many are correct?")
print("- Recall: of true annotated objects, how many were found?")
print("- mAP50: mean average precision at IoU threshold 0.50.")
print("- mAP50-95: mean average precision averaged across IoU thresholds 0.50 to 0.95.")
print("- A large gap between mAP50 and mAP50-95 can indicate boxes are not tightly localized.")
print("- Review confusion matrix, PR curves, and predicted/label visualizations in the evaluation run directory.")

## 14. Visualize predictions on test images

In [ ]:
test_image_dir = DATASET_DIR / "images" / "test"
test_images = sorted(p for p in test_image_dir.iterdir()
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

if not test_images:
    print("No test images found.")
else:
    prediction_samples = random.sample(test_images, min(6, len(test_images)))
    prediction_results = best_model.predict(
        source=[str(p) for p in prediction_samples],
        imgsz=IMAGE_SIZE,
        conf=0.25,
        device=DEVICE,
        save=True,
        project="runs/brain_tumor",
        name="test_predictions",
        exist_ok=True
    )
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax in axes.flat:
        ax.axis("off")
    for ax, result in zip(axes.flat, prediction_results):
        plotted = result.plot()  # BGR image with predicted boxes
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(Path(result.path).name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 15. Save artifacts and a reproducibility manifest

In [ ]:
ARTIFACT_DIR = Path("./artifacts/brain_tumor_yolov8")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if BEST_WEIGHTS.exists():
    shutil.copy2(BEST_WEIGHTS, ARTIFACT_DIR / "best.pt")
if LAST_WEIGHTS.exists():
    shutil.copy2(LAST_WEIGHTS, ARTIFACT_DIR / "last.pt")
shutil.copy2(data_yaml_path, ARTIFACT_DIR / "data.yaml")

manifest = {
    "model_weights": MODEL_WEIGHTS,
    "best_checkpoint": str((ARTIFACT_DIR / "best.pt").resolve()),
    "class_names": CLASS_NAMES,
    "image_size": IMAGE_SIZE,
    "epochs_requested": EPOCHS,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
    "seed": SEED,
    "test_metrics": summary_metrics if "summary_metrics" in globals() else None,
    "ultralytics_version": ultralytics.__version__,
}
import json
with open(ARTIFACT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Artifacts saved to:", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(p.name, "-", p.stat().st_size, "bytes")

## 16. Optional: export to ONNX

Use this if you need to deploy the model in an environment that does not load PyTorch checkpoints. Keep `best.pt` as the primary Ultralytics checkpoint.

In [ ]:
# Uncomment to export:
# exported_path = best_model.export(format="onnx", imgsz=IMAGE_SIZE)
# print("Exported model:", exported_path)

## Final checklist

- [ ] `SOURCE_IMAGES`, `SOURCE_LABELS`, and `CLASS_NAMES` match your actual dataset.
- [ ] All displayed bounding boxes were visually checked.
- [ ] Annotation class IDs match the intended class names.
- [ ] No patient-level leakage exists between train, validation, and test (where patient IDs are available).
- [ ] Test metrics were calculated on the held-out test split, not the training or validation split.
- [ ] False positives and false negatives were inspected, especially missed tumors.
- [ ] The saved `best.pt`, `data.yaml`, and `manifest.json` are preserved.

**Medical-use note:** this is an educational research workflow, not a clinically validated diagnostic system. Report dataset limitations and avoid interpreting model predictions as a medical diagnosis.